# Description

In this notebook, we benchmark EQL with division algorithm on the set of previously generated SR benchmarks.

In [1]:
from __future__ import annotations

import csv
import time
from pathlib import Path

import h5py
import numpy as np
import sympy as sp

from config.benchmark_config import DataCFG, EQLDIV
from src.EQLdiv.mlfg_final import test_mlfg


# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------

def _mse_np(yhat, y):
    yhat = np.asarray(yhat).reshape(-1)
    y = np.asarray(y).reshape(-1)
    return float(np.mean((yhat - y) ** 2))


def _load_one_group(f: h5py.File, gname: str):
    g = f[gname]

    raw = g["sympy_str"][()]
    true_expr_str = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else str(raw)

    Xtr = g["train"]["X"][...].astype(np.float32)
    ytr = g["train"]["y"][...].astype(np.float32).reshape(-1, 1)

    Xti = g["test_interp"]["X"][...].astype(np.float32)
    yti = g["test_interp"]["y"][...].astype(np.float32).reshape(-1, 1)

    Xte = g["test_extrap"]["X"][...].astype(np.float32)
    yte = g["test_extrap"]["y"][...].astype(np.float32).reshape(-1, 1)

    return true_expr_str, Xtr, ytr, Xti, yti, Xte, yte


def _complexity_from_expr(expr):
    try:
        return int(sp.count_ops(expr, visual=False))
    except Exception:
        return -1


# ------------------------------------------------------------
# Main experiment
# ------------------------------------------------------------

def main():
    cfg = DataCFG()
    out_csv = Path(EQLDIV.results_path)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    with h5py.File(cfg.h5_path, "r") as f, out_csv.open("w", newline="") as out:
        w = csv.writer(out)
        w.writerow([
            "group",
            "run",
            "seed",
            "train_mse",
            "test_interp_mse",
            "test_extrap_mse",
            "complexity",
            "num_active",
            "duration_s",
            "true_expr",
            "found_expr",
        ])

        for gname in sorted(f.keys()):
            true_expr_str, Xtr, ytr, Xti, yti, Xte, yte = _load_one_group(f, gname)

            for run in range(EQLDIV.n_runs):
                seed = EQLDIV.base_seed + run
                np.random.seed(seed)

                # Apply EQL-div exactly in the same way as in the working notebook:
                # datasets = (train, validation, test)
                # here:
                #   train       -> original train
                #   validation  -> test_interp
                #   test        -> test_extrap
                datasets = (
                    (Xtr, ytr),
                    (Xti, yti),
                    (Xte, yte),
                )

                t0 = time.perf_counter()

                trained = test_mlfg(
                    datasets=datasets,
                    n_epochs=EQLDIV.n_epochs,
                    verbose=EQLDIV.verbose,
                    learning_rate=EQLDIV.learning_rate,
                    basefuncs1=list(EQLDIV.basefuncs1),
                    basefuncs2=list(EQLDIV.basefuncs2),
                    L1_reg=EQLDIV.L1_reg,
                    L2_reg=EQLDIV.L2_reg,
                    reg_start=EQLDIV.reg_start,
                    reg_end=EQLDIV.reg_end,
                    batch_size=EQLDIV.batch_size,
                    n_layer=EQLDIV.n_layer,
                    n_per_base=EQLDIV.n_per_base,
                    classifier=None,
                    gradient=EQLDIV.gradient,
                    init_state=None,
                    validate_every=EQLDIV.validate_every,
                    k=EQLDIV.k,
                    id=seed,
                )

                dur = time.perf_counter() - t0

                classifier = trained["classifier"]

                yhat_tr = classifier.evaluate(Xtr)
                yhat_ti = classifier.evaluate(Xti)
                yhat_te = classifier.evaluate(Xte)

                train_mse = _mse_np(yhat_tr, ytr)
                test_interp_mse = _mse_np(yhat_ti, yti)
                test_extrap_mse = _mse_np(yhat_te, yte)

                try:
                    expr = classifier.get_symbolic_expression(
                        thresh=EQLDIV.symbolic_prune_threshold,
                        simplify_expr=EQLDIV.simplify,
                    )
                    expr_str = str(expr)
                    complexity = _complexity_from_expr(expr)
                except Exception as e:
                    expr_str = f"<symbolic extraction failed: {e}>"
                    complexity = -1

                num_active = classifier.get_num_active_units(thresh=EQLDIV.activity_threshold)

                w.writerow([
                    gname,
                    run,
                    seed,
                    train_mse,
                    test_interp_mse,
                    test_extrap_mse,
                    complexity,
                    num_active,
                    dur,
                    true_expr_str,
                    expr_str,
                ])
                out.flush()

                print(
                    f"[{gname}] run={run} seed={seed} "
                    f"train={train_mse:.3e} interp={test_interp_mse:.3e} "
                    f"extrap={test_extrap_mse:.3e} active={num_active} complexity={complexity}"
                )

    print(f"\nSaved: {out_csv}")


if __name__ == "__main__":
    main()

Max input value is:  1.96138
... building the model
... training
Epoch:  500 	Best val error:  0.0007135190280678216 	current val error:  0.0007135190280678216
Epoch:  1000 	Best val error:  0.00013812873953611415 	current val error:  0.00013812873953611415
Epoch:  1500 	Best val error:  3.994077701463539e-05 	current val error:  3.994077701463539e-05
Epoch:  2000 	Best val error:  1.8592202977174566e-05 	current val error:  1.8592202977174566e-05
Epoch:  2500 	Best val error:  1.2899030881641238e-05 	current val error:  1.3004216924628054e-05
Epoch:  3000 	Best val error:  9.84549118854261e-06 	current val error:  9.84549118854261e-06
Epoch:  3500 	Best val error:  8.143433159801816e-06 	current val error:  8.143433159801816e-06
Epoch:  4000 	Best val error:  6.993313359515696e-06 	current val error:  6.993313359515696e-06
Epoch:  4500 	Best val error:  6.04583486563115e-06 	current val error:  6.04583486563115e-06
Epoch:  5000 	Best val error:  5.311600199675581e-06 	current val erro

The code for file mlfg_final.py ran for 0.23m


Epoch:  500 	Best val error:  5.974378854034512e-05 	current val error:  5.974378854034512e-05
Epoch:  1000 	Best val error:  4.465840402190224e-05 	current val error:  7.706240126026387e-05
Epoch:  1500 	Best val error:  3.129844172633511e-05 	current val error:  3.129844172633511e-05
Epoch:  2000 	Best val error:  2.3837238373403125e-05 	current val error:  2.3837238373403125e-05
Epoch:  2500 	Best val error:  2.055948588974843e-05 	current val error:  2.0618467267752294e-05
Epoch:  3000 	Best val error:  1.6613590304359604e-05 	current val error:  1.6613590304359604e-05
Epoch:  3500 	Best val error:  1.3163853303410633e-05 	current val error:  1.3163853303410633e-05
Epoch:  4000 	Best val error:  1.2777983180001229e-05 	current val error:  1.2977776627565163e-05
Epoch:  4500 	Best val error:  9.318159097659873e-06 	current val error:  9.318159097659873e-06
Epoch:  5000 	Best val error:  7.685293571313423e-06 	current val error:  7.685293571313423e-06
Epoch:  5500 	Best val error:  6

The code for file mlfg_final.py ran for 0.22m


Epoch:  500 	Best val error:  0.00045999672829566407 	current val error:  0.00045999672829566407
Epoch:  1000 	Best val error:  2.8377299372550624e-05 	current val error:  2.8377299372550624e-05
Epoch:  1500 	Best val error:  1.3106304390220203e-05 	current val error:  1.3106304390220203e-05
Epoch:  2000 	Best val error:  7.5032678275022135e-06 	current val error:  7.5032678275022135e-06
Epoch:  2500 	Best val error:  5.785624827225888e-06 	current val error:  5.785624827225888e-06
Epoch:  3000 	Best val error:  3.6906782057144483e-06 	current val error:  4.500948863395138e-06
Epoch:  3500 	Best val error:  2.5610437406342612e-06 	current val error:  2.644861125666864e-06
Epoch:  4000 	Best val error:  2.0249740435573926e-06 	current val error:  2.438377593172447e-06
Epoch:  4500 	Best val error:  1.481953214810261e-06 	current val error:  1.481953214810261e-06
Epoch:  5000 	Best val error:  1.3977115798091688e-06 	current val error:  1.521013072380839e-06
Epoch:  5500 	Best val error:

...

The code for file mlfg_final.py ran for 0.23m


Epoch:  500 	Best val error:  46.565597160719335 	current val error:  631.1729375170544
Epoch:  1000 	Best val error:  38.99805324198678 	current val error:  38.99805324198678
Epoch:  1500 	Best val error:  38.99805324198678 	current val error:  471.3396965810098
Epoch:  2000 	Best val error:  33.51065834343899 	current val error:  2566.543720759684
Epoch:  2500 	Best val error:  32.40139326872304 	current val error:  32.40139326872304
Epoch:  3000 	Best val error:  32.40139326872304 	current val error:  37.300563253927976
Epoch:  3500 	Best val error:  32.40139326872304 	current val error:  44.00443857861683
Epoch:  4000 	Best val error:  32.40139326872304 	current val error:  39.844704693648964
Epoch:  4500 	Best val error:  32.40139326872304 	current val error:  46.109870690852404
Epoch:  5000 	Best val error:  32.40139326872304 	current val error:  46.50488625653088
Epoch:  5500 	Best val error:  32.40139326872304 	current val error:  46.12579771876335
Epoch:  6000 	Best val error:

The code for file mlfg_final.py ran for 0.23m


Epoch:  500 	Best val error:  38.22245999868028 	current val error:  39.52705606352538
Epoch:  1000 	Best val error:  38.22245999868028 	current val error:  730.6485352050513
Epoch:  1500 	Best val error:  38.22245999868028 	current val error:  40.01606914168224
Epoch:  2000 	Best val error:  37.81074679084122 	current val error:  39.653157064225525
Epoch:  2500 	Best val error:  37.81074679084122 	current val error:  40.46703775692731
Epoch:  3000 	Best val error:  37.81074679084122 	current val error:  2884.056133895647
Epoch:  3500 	Best val error:  37.81074679084122 	current val error:  2548.013960758224
Epoch:  4000 	Best val error:  36.57605200074613 	current val error:  698.0408703498542
Epoch:  4500 	Best val error:  36.57605200074613 	current val error:  43.80934415291995
Epoch:  5000 	Best val error:  36.57605200074613 	current val error:  8528.389551717788
Epoch:  5500 	Best val error:  36.57605200074613 	current val error:  21851.78915610537
Epoch:  6000 	Best val error:  3

The code for file mlfg_final.py ran for 0.23m


In [2]:
from __future__ import annotations

import math
from pathlib import Path

import pandas as pd
import sympy as sp

from config.benchmark_config import EQLDIV


def _safe_sympify(expr_str: str) -> sp.Expr | None:
    if not isinstance(expr_str, str) or not expr_str.strip():
        return None
    if expr_str.startswith("<") and expr_str.endswith(">"):
        return None
    try:
        return sp.sympify(expr_str)
    except Exception:
        return None


def _safe_latex(expr_str: str) -> str | None:
    expr = _safe_sympify(expr_str)
    if expr is None:
        return None
    try:
        return sp.latex(expr)
    except Exception:
        return None


def _format_pm(value: float, std: float, sig: int = 1) -> str:
    if not math.isfinite(value) or not math.isfinite(std):
        return "N/A"

    if value == 0.0:
        return r"(0\pm0)\times 10^{0}"

    exp = int(math.floor(math.log10(abs(value))))
    scale = 10 ** exp

    v = round(value / scale, sig)
    s = round(std / scale, sig)

    return rf"({v}\pm{s})\times 10^{{{exp}}}"


def _count_nodes(expr: sp.Expr) -> int:
    return sum(1 for _ in sp.preorder_traversal(expr))


def summarize(csv_path: str | Path, k: int = 5) -> None:
    df = pd.read_csv(csv_path)

    metrics = [
        "train_mse",
        "test_interp_mse",
        "test_extrap_mse",
    ]

    for gname, gdf in df.groupby("group"):
        print(f"\n{gname}")

        gdf = gdf.sort_values("test_extrap_mse").iloc[:k]

        for m in metrics:
            if m not in gdf.columns:
                print(f"  {m}: N/A")
                continue

            vals = pd.to_numeric(gdf[m], errors="coerce").dropna().to_numpy()

            if len(vals) == 0:
                print(f"  {m}: N/A")
                continue

            mean = float(vals.mean())
            std = float(vals.std(ddof=0))

            print(f"  {m}: {_format_pm(mean, std)}")

        node_counts = []

        for s in gdf["found_expr"]:
            expr = _safe_sympify(s)
            if expr is None:
                continue
            try:
                node_counts.append(_count_nodes(expr))
            except Exception:
                pass

        if node_counts:
            nc = pd.Series(node_counts, dtype=float)
            print(f"  node_count: {nc.mean():.1f} ± {nc.std(ddof=0):.1f}")
        else:
            print("  node_count: N/A")

        best_row = gdf.sort_values("train_mse").iloc[0]

        latex_expr = _safe_latex(best_row["found_expr"])
        if latex_expr is not None:
            print("  best_train_expr_latex:")
            print(f"    ${latex_expr}$")
        else:
            print("  best_train_expr_latex: N/A")

        print("  best_train_expr_raw:")
        print(f"    {best_row['found_expr']}")


if __name__ == "__main__":
    summarize(EQLDIV.results_path, k=5)


expr_000_lin_uni
  train_mse: (7.8\pm6.0)\times 10^{-7}
  test_interp_mse: (8.2\pm6.2)\times 10^{-7}
  test_extrap_mse: (5.3\pm8.0)\times 10^{-3}
  node_count: 37.0 ± 5.8
  best_train_expr_latex:
    $\frac{- 8.96215494168 \cdot 10^{-7} x_{1}^{4} + 0.000994174726522 x_{1}^{3} - 0.12825345014 x_{1}^{2} + 0.765215736862 x_{1} + 0.971661631996}{1.97134569161 \cdot 10^{-8} x_{1}^{4} - 1.33793449675 \cdot 10^{-5} x_{1}^{3} + 0.000520766639823 x_{1}^{2} - 0.0690615222035 x_{1} + 0.483435508061}$
  best_train_expr_raw:
    (-8.96215494168e-7*x1**4 + 0.000994174726522*x1**3 - 0.12825345014*x1**2 + 0.765215736862*x1 + 0.971661631996)/(1.97134569161e-8*x1**4 - 1.33793449675e-5*x1**3 + 0.000520766639823*x1**2 - 0.0690615222035*x1 + 0.483435508061)

expr_001_lin_bi
  train_mse: (6.6\pm7.1)\times 10^{-3}
  test_interp_mse: (5.7\pm6.4)\times 10^{-3}
  test_extrap_mse: (2.9\pm5.4)\times 10^{-1}
  node_count: 115.4 ± 38.1
  best_train_expr_latex:
    $\frac{0.193183252558 x_{1}^{2} + 0.448833921402 x